# FreshMart Lab 2: Feature Engineering
**Azure Machine Learning (ทางเลือกฉุกเฉินแทน Fabric)**

เป้าหมาย: (A) ฝึกแปลงข้อมูลแบบเห็นผล  (B) เตรียมฟีเจอร์สมาชิกให้ Lab 3–4

บน Azure ML Studio notebook **ไม่มี Data Wrangler ของ Fabric**  
ส่วน A จึงใช้ pandas ให้เห็นผลเดียวกัน — ถ้าเปิด notebook ใน VS Code จะใช้ส่วนขยาย Data Wrangler แทนได้

### กฎทองที่ต้องจำ (ลดคำถาม Lab 4 ครึ่งหนึ่ง)
โมเดลเรียนรู้จาก **min/max ของชุดฝึก 1,500 คน**  
Lab 4 ทำนายชุดใหม่ 200 คน — **ห้ามคำนวณสเกลใหม่** ต้องใช้ `feature_params.json` จากแล็บนี้

### ถ้าติด — อ่านก่อนถาม TA
| อาการ | ทำอะไร |
| --- | --- |
| หา Data Wrangler ใน Studio ไม่เจอ | ปกติของแทร็กนี้ — ใช้เซลล์ pandas ในส่วน A |
| Export จาก VS Code แล้วสับสน | ส่วน A เป็นการฝึกแปลงข้อมูล — **ส่วน B ต้องรันเซลล์ที่เตรียมให้** เพื่อส่งงาน |


### เตรียมฟังก์ชันโหลด

รันเซลล์ถัดไปก่อนเสมอ


In [ ]:
import json
from pathlib import Path
import pandas as pd

from pathlib import Path
import pandas as pd

BRONZE_TRANSACTIONS = "bronze/transactions"
BRONZE_CUSTOMERS = "bronze/customers"
SILVER_CUSTOMER_FEATURES = "silver/customer_features"
GOLD_PREDICTIONS = "gold/freshmart_predictions"
ASSET_BRONZE_TRANSACTIONS = "bronze-transactions"
ASSET_BRONZE_CUSTOMERS = "bronze-customers"
ASSET_SCORING_BATCH = "scoring-batch"
ASSET_SILVER_FEATURES = "silver-customer-features"
ASSET_GOLD_PREDICTIONS = "gold-freshmart-predictions"
TABLE_TO_ASSET = {
    BRONZE_TRANSACTIONS: ASSET_BRONZE_TRANSACTIONS,
    BRONZE_CUSTOMERS: ASSET_BRONZE_CUSTOMERS,
    SILVER_CUSTOMER_FEATURES: ASSET_SILVER_FEATURES,
    GOLD_PREDICTIONS: ASSET_GOLD_PREDICTIONS,
}
CSV_TO_ASSET = {
    "freshmart_transactions.csv": ASSET_BRONZE_TRANSACTIONS,
    "freshmart_customers.csv": ASSET_BRONZE_CUSTOMERS,
    "freshmart_scoring_batch.csv": ASSET_SCORING_BATCH,
}
RAW_FILES = (
    "freshmart_transactions.csv",
    "freshmart_customers.csv",
    "freshmart_scoring_batch.csv",
)


def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None


def _writable_dir(path: Path) -> Path | None:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return path.resolve()
    except OSError:
        return None


def resolve_raw_csv(file_name: str) -> Path:
    found = _first_existing([
        Path("data") / "raw" / file_name,
        Path("data") / file_name,
        Path("../data") / "raw" / file_name,
        Path("../data") / file_name,
        Path("../../labs/data") / file_name,
        Path("../labs/data") / file_name,
        Path("labs/data") / file_name,
        Path(file_name),
    ])
    if found is None:
        raise FileNotFoundError(
            f"Cannot find {file_name}. Upload it to data/raw/ next to the notebook "
            "or clone this repo so labs/data/ is available."
        )
    return found


def load_csv(file_name: str) -> pd.DataFrame:
    asset_name = CSV_TO_ASSET.get(file_name)
    if asset_name:
        frame = load_data_asset(asset_name)
        if frame is not None:
            return frame
    found = resolve_raw_csv(file_name)
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)


def load_data_asset(asset_name: str):
    """Load a registered Azure ML data asset, or return None."""
    try:
        from azure.ai.ml import MLClient
        from azure.identity import DefaultAzureCredential

        ml_client = MLClient.from_config(credential=DefaultAzureCredential())
        asset = ml_client.data.get(name=asset_name, label="latest")
        path = asset.path
        print(f"Loaded data asset {asset_name} v{asset.version}: {path}")
        if str(path).lower().endswith(".parquet"):
            return pd.read_parquet(path)
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Data asset '{asset_name}' unavailable ({exc})")
        return None


def register_data_asset(name: str, path, description: str = "") -> bool:
    """Register a file as an Azure ML data asset. Returns True when saved."""
    try:
        from azure.ai.ml import MLClient
        from azure.ai.ml.entities import Data
        from azure.ai.ml.constants import AssetTypes
        from azure.identity import DefaultAzureCredential

        ml_client = MLClient.from_config(credential=DefaultAzureCredential())
        asset = Data(
            name=name,
            path=str(path),
            type=AssetTypes.URI_FILE,
            description=description or f"FreshMart {name}",
        )
        created = ml_client.data.create_or_update(asset)
        print(f"Registered data asset {created.name} v{created.version}")
        return True
    except Exception as exc:
        print(f"Could not register data asset '{name}' ({exc})")
        return False


def resolve_artifact_root() -> Path:
    for candidate in (
        Path("data"),
        Path("../data"),
        Path("labs-azureml/data"),
    ):
        ready = _writable_dir(candidate)
        if ready is not None:
            return ready
    fallback = Path("data")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback.resolve()


ARTIFACT_ROOT = resolve_artifact_root()


def layer_path(layer: str, stem: str, suffix: str = ".parquet") -> Path:
    folder = ARTIFACT_ROOT / layer
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{stem}{suffix}"


def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    asset_name = TABLE_TO_ASSET.get(table_name)
    if asset_name:
        frame = load_data_asset(asset_name)
        if frame is not None:
            return frame
    layer, _, stem = table_name.partition("/")
    parquet = layer_path(layer, stem)
    if parquet.exists():
        frame = pd.read_parquet(parquet)
        print(f"Loaded parquet {parquet}: {len(frame):,} rows")
        return frame
    csv_fallback = ARTIFACT_ROOT / layer / f"{stem}.csv"
    if csv_fallback.exists():
        frame = pd.read_csv(csv_fallback)
        print(f"Loaded CSV artifact {csv_fallback}: {len(frame):,} rows")
        return frame
    print(f"Layer file '{table_name}' not found. Falling back to published CSV.")
    return load_csv(file_name)


def save_layer(frame: pd.DataFrame, table_name: str) -> Path:
    layer, _, stem = table_name.partition("/")
    parquet = layer_path(layer, stem)
    try:
        frame.to_parquet(parquet, index=False)
        print(f"Wrote {parquet} ({len(frame):,} rows)")
        return parquet
    except Exception as exc:
        csv_path = layer_path(layer, stem, suffix=".csv")
        frame.to_csv(csv_path, index=False)
        print(f"Parquet unavailable ({exc}). Wrote {csv_path}")
        return csv_path


### ส่วน A — โหลดธุรกรรมสำหรับฝึกแปลงข้อมูล

**โค้ดนี้ทำอะไร:** โหลดธุรกรรมแล้วสุ่ม **500 แถว** ให้ทดลองกรองและรวมยอดได้เร็ว

ตัวแปรสำคัญ: **`df`**


In [ ]:
df = load_table_or_csv(
    BRONZE_TRANSACTIONS,
    "freshmart_transactions.csv",
)
df = df.sample(n=500, random_state=1).reset_index(drop=True)
print("Sample for Data Wrangler:", df.shape)
df.head(4)


### ส่วน A — จัดข้อความ กรอง และรวมยอด (เทียบเท่า Data Wrangler)

ทำตามทีละข้อในเซลล์ด้านล่าง หรือเปิด VS Code แล้วใช้ Data Wrangler กับ `df` แล้ววางโค้ดแทน

1. จัดรูปแบบ `Category` ให้ขึ้นต้นด้วยตัวพิมพ์ใหญ่
2. กรอง `StoreType = Express` แล้วเรียง `WasteCost` จากมากไปน้อย
3. รวมยอดเฉลี่ย `WasteCost` ตาม `Category`


In [ ]:
df_formatted = df.copy()
df_formatted["Category"] = df_formatted["Category"].astype(str).str.title()

express = (
    df_formatted[df_formatted["StoreType"] == "Express"]
    .sort_values("WasteCost", ascending=False)
)
print("Top Express waste rows:")
display(express.head(5))


def summarize_waste(frame: pd.DataFrame) -> pd.DataFrame:
    """เทียบเท่า Group by จาก Data Wrangler — ปรับได้ถ้าคุณ export โค้ดจาก VS Code"""
    return frame.groupby(["Category"]).agg(WasteCost_mean=("WasteCost", "mean")).reset_index()


print(summarize_waste(df_formatted))


### ส่วน B — โหลดสมาชิกสำหรับโมเดล

**ทำไมแยกจากส่วน A:** ส่วน A ฝึกกับธุรกรรม ส่วนโมเดลทำนาย Churn ใช้ตารางสมาชิก

**สิ่งที่ควรเห็น:** shape ประมาณ `(1500, 11)` และ Age ว่าง **37** แถว

**ส่งงาน Lab 3–4 ต้องรันเซลล์สัญญาฟีเจอร์ด้านล่างเท่านั้น**


In [ ]:
df_cust = load_table_or_csv(BRONZE_CUSTOMERS, "freshmart_customers.csv")
print("Customers DataFrame loaded:", df_cust.shape)
print("Age missing:", int(df_cust["Age"].isna().sum()))
df_cust.head(3)


### ส่วน B — สัญญาฟีเจอร์ (ความรู้สั้น ๆ)

เซลล์ถัดไปฝังฟังก์ชัน `fit_preprocessor` / `transform_customers` ให้แล้ว

| ขั้น | ความหมาย |
| --- | --- |
| เติม Age ด้วย median | ชุดนี้ median = **37.0** |
| One-hot | แปลง `MembershipTier` / `Gender` เป็นคอลัมน์ 0/1 |
| Min-max scale | ปรับเงินและความถี่ให้อยู่ช่วงประมาณ 0–1 จาก**ชุดฝึก** |

**รันเซลล์ฟังก์ชันก่อน** แล้วค่อยรันเซลล์ `fit` / `transform`


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any
import logging

logger = logging.getLogger(__name__)

ID_COLUMN = 'CustomerID'
TARGET_COLUMN = 'Churn'
CATEGORICAL_COLUMNS = ('MembershipTier', 'Gender')
IMPUTE_MEDIAN_COLUMNS = ('Age',)
SCALE_COLUMNS = ('MonetaryTotal', 'AvgBasketSize', 'RecencyDays', 'TenureMonths')
DUMMY_COLUMNS = ('MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
PASSTHROUGH_COLUMNS = ('Frequency', 'ComplaintCount')
FEATURE_COLUMNS = ('Age', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount', 'MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
REQUIRED_RAW_COLUMNS = ('CustomerID', 'Age', 'Gender', 'MembershipTier', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount')

@dataclass(frozen=True)
class FeatureParams:
    """Fitted preprocessing parameters reused at scoring time.

    Attributes:
        age_median: Median Age computed from the training customers.
        scale_mins: Per-column minimum used for min-max scaling.
        scale_maxs: Per-column maximum used for min-max scaling.
        dummy_columns: One-hot columns the model expects, in order.
        feature_columns: Final model input columns, in order.
    """

    age_median: float
    scale_mins: dict[str, float]
    scale_maxs: dict[str, float]
    dummy_columns: list[str]
    feature_columns: list[str]

    def to_dict(self) -> dict[str, Any]:
        """Serialize parameters to a JSON-friendly dictionary."""
        return asdict(self)

    @classmethod
    def from_dict(cls, payload: dict[str, Any]) -> FeatureParams:
        """Create parameters from a dictionary.

        Args:
            payload: Mapping produced by ``to_dict``.

        Returns:
            Validated ``FeatureParams``.

        Raises:
            ValueError: If required keys are missing.
        """
        required = {
            "age_median",
            "scale_mins",
            "scale_maxs",
            "dummy_columns",
            "feature_columns",
        }
        missing = required - set(payload)
        if missing:
            raise ValueError(f"FeatureParams missing keys: {sorted(missing)}")
        return cls(
            age_median=float(payload["age_median"]),
            scale_mins={k: float(v) for k, v in payload["scale_mins"].items()},
            scale_maxs={k: float(v) for k, v in payload["scale_maxs"].items()},
            dummy_columns=list(payload["dummy_columns"]),
            feature_columns=list(payload["feature_columns"]),
        )


def validate_raw_customers(df: pd.DataFrame, *, require_target: bool = True) -> pd.DataFrame:
    """Validate and normalize a raw FreshMart customer frame.

    Args:
        df: Raw customer records from CSV or ``bronze.customers``.
        require_target: When True, require the ``Churn`` column.

    Returns:
        Copy with numeric columns coerced.

    Raises:
        ValueError: If required columns are missing.
    """
    required = REQUIRED_RAW_COLUMNS + ((TARGET_COLUMN,) if require_target else ())
    _require_columns(df, required, frame_name="customer frame")
    numeric_cols = (
        "Age",
        "TenureMonths",
        "RecencyDays",
        "Frequency",
        "MonetaryTotal",
        "AvgBasketSize",
        "ComplaintCount",
    )
    if require_target:
        numeric_cols = numeric_cols + (TARGET_COLUMN,)
    return _numeric_copy(df, numeric_cols)


def _require_columns(df: pd.DataFrame, columns: tuple[str, ...], *, frame_name: str) -> None:
    """Raise if expected columns are missing."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{frame_name} missing required columns: {missing}")


def _numeric_copy(df: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Return a copy with selected columns coerced to numeric."""
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def fit_preprocessor(df_raw: pd.DataFrame) -> FeatureParams:
    """Fit imputation and scaling parameters on training customers.

    Args:
        df_raw: Raw customer DataFrame including ``Churn``.

    Returns:
        Fitted parameters that must be reused for scoring.
    """
    df = validate_raw_customers(df_raw, require_target=True)
    age_median = float(df["Age"].median())
    if pd.isna(age_median):
        raise ValueError("Cannot fit preprocessor: Age median is NaN")

    scale_mins: dict[str, float] = {}
    scale_maxs: dict[str, float] = {}
    for col in SCALE_COLUMNS:
        col_min = float(df[col].min())
        col_max = float(df[col].max())
        if col_max <= col_min:
            raise ValueError(f"Cannot scale {col}: min={col_min}, max={col_max}")
        scale_mins[col] = col_min
        scale_maxs[col] = col_max

    params = FeatureParams(
        age_median=age_median,
        scale_mins=scale_mins,
        scale_maxs=scale_maxs,
        dummy_columns=list(DUMMY_COLUMNS),
        feature_columns=list(FEATURE_COLUMNS),
    )
    logger.info(
        "Fitted feature params: age_median=%.1f, scale_columns=%s",
        params.age_median,
        list(SCALE_COLUMNS),
    )
    return params


def _one_hot_categories(df: pd.DataFrame) -> pd.DataFrame:
    """One-hot encode membership and gender, keeping the expected columns."""
    encoded = pd.get_dummies(df, columns=list(CATEGORICAL_COLUMNS), drop_first=False)
    for col in DUMMY_COLUMNS:
        if col not in encoded.columns:
            encoded[col] = 0
    return encoded


def _min_max_scale(series: pd.Series, col_min: float, col_max: float) -> pd.Series:
    """Scale a series to [0, 1] using fitted min/max."""
    return (series - col_min) / (col_max - col_min + 1e-6)


def transform_customers(
    df_raw: pd.DataFrame,
    params: FeatureParams,
    *,
    require_target: bool = False,
) -> pd.DataFrame:
    """Apply the fitted FreshMart preprocessing contract.

    Args:
        df_raw: Raw customer records (training or scoring batch).
        params: Parameters from ``fit_preprocessor``.
        require_target: When True, keep and validate ``Churn``.

    Returns:
        Feature frame with ``CustomerID``, model columns, and optional ``Churn``.
    """
    df = validate_raw_customers(df_raw, require_target=require_target)
    work = df.copy()
    work["Age"] = work["Age"].fillna(params.age_median)

    work = _one_hot_categories(work)
    for col in SCALE_COLUMNS:
        work[col] = _min_max_scale(
            work[col],
            params.scale_mins[col],
            params.scale_maxs[col],
        )

    bool_cols = work.select_dtypes(include="bool").columns
    work[bool_cols] = work[bool_cols].astype(int)

    ordered = [ID_COLUMN, *params.feature_columns]
    if require_target or TARGET_COLUMN in work.columns:
        ordered.append(TARGET_COLUMN)
    missing = [col for col in ordered if col not in work.columns]
    if missing:
        raise ValueError(f"Transformed frame missing columns: {missing}")

    result = work[ordered].copy()
    feature_frame = result[list(params.feature_columns)]
    if feature_frame.isna().any().any():
        bad = feature_frame.columns[feature_frame.isna().any()].tolist()
        raise ValueError(f"NaN remaining in feature columns: {bad}")
    return result


def model_matrix(df_features: pd.DataFrame, params: FeatureParams) -> pd.DataFrame:
    """Return the model input matrix in signature order.

    Args:
        df_features: Output of ``transform_customers``.
        params: Fitted parameters.

    Returns:
        DataFrame with only model feature columns.
    """
    _require_columns(df_features, tuple(params.feature_columns), frame_name="feature frame")
    return df_features[list(params.feature_columns)].astype(float)


def save_feature_params(params: FeatureParams, path: str | Path) -> Path:
    """Write feature parameters to a JSON file.

    Args:
        params: Fitted parameters.
        path: Destination JSON path.

    Returns:
        Resolved output path.
    """
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(params.to_dict(), indent=2), encoding="utf-8")
    logger.info("Saved feature params to %s", out)
    return out


def load_feature_params(path: str | Path) -> FeatureParams:
    """Load feature parameters from a JSON file.

    Args:
        path: JSON path written by ``save_feature_params``.

    Returns:
        Fitted parameters.

    Raises:
        FileNotFoundError: If the file does not exist.
    """
    src = Path(path)
    if not src.exists():
        raise FileNotFoundError(f"Feature params not found: {src}")
    payload = json.loads(src.read_text(encoding="utf-8"))
    return FeatureParams.from_dict(payload)


### fit แล้ว transform

**โค้ดนี้ทำอะไร**
1. `fit_preprocessor` จำ median/min/max จากชุดฝึก
2. `transform_customers` สร้างตารางฟีเจอร์พร้อมใช้

ตรวจ Age median ≈ 37.0 และคอลัมน์ one-hot ครบ


In [ ]:
params = fit_preprocessor(df_cust)
df_clean = transform_customers(df_cust, params, require_target=True)
print(f"Cleaned DataFrame Shape: {df_clean.shape}")
print("Feature columns:", params.feature_columns)
print(f"Age median used for imputation: {params.age_median:.1f}")
display(df_clean.head(5))


### บันทึก Silver + params

**โค้ดนี้ทำอะไร**
- เขียน `data/params/feature_params.json` (Lab 4 ต้องใช้ไฟล์นี้)
- เขียน `data/silver/customer_features.parquet`

**สิ่งที่ควรเห็น:** เซลล์พิมพ์เส้นทางไฟล์ที่บันทึก และ `Lab 2 verification passed`


In [ ]:
params_dir = ARTIFACT_ROOT / "params"
params_dir.mkdir(parents=True, exist_ok=True)
params_path = params_dir / "feature_params.json"
save_feature_params(params, params_path)
print(f"Saved {params_path}")

silver_path = save_layer(df_clean, SILVER_CUSTOMER_FEATURES)
silver_csv = layer_path("silver", "customer_features", suffix=".csv")
df_clean.to_csv(silver_csv, index=False)
print(f"Saved silver features: {silver_path}")
print(f"Saved AutoML tabular CSV: {silver_csv}")
register_data_asset(ASSET_SILVER_FEATURES, silver_csv, "FreshMart silver customer features for AutoML")


### จุดตรวจ Lab 2

ผ่านแล้วพิมพ์ `Lab 2 verification passed`  
ในโฟลเดอร์ Files ควรเห็น `data/silver/customer_features.parquet`


In [ ]:
if df_clean["Age"].isna().sum() != 0:
    raise AssertionError("หลังเตรียมฟีเจอร์แล้ว Age ไม่ควรว่าง")
if not set(["MembershipTier_Bronze", "Gender_F"]).issubset(df_clean.columns):
    raise AssertionError("ขาดคอลัมน์ one-hot ที่โมเดลคาดหวัง — รันเซลล์ fit/transform อีกครั้ง")
if df_clean["MonetaryTotal"].max() > 1.000001:
    raise AssertionError("MonetaryTotal ควรอยู่ในช่วงประมาณ 0–1 หลัง scale จากชุดฝึก")
print("Lab 2 verification passed")
